In [120]:
%load_ext autoreload
%autoreload 2
import sys
import pandas as pd
from pathlib import Path
import numpy as np

sys.path.insert(0, str(Path().resolve().parents[0]))
from transform.clean import normalize_paint, merge_variants 

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [78]:
INFOBOX_FIELDS = ["kit_name","image","categories","franchise","run","release date", "materials", "scale", "classification", 
                  "image_url", "price", "need glue?", "japanese name", "model of", "jan/isbn", "lineup no.", "variant of", "subtitle", "need to paint", "illustration by",
                  "need paint?","exclusive to", "imgsize"]

PAINT_COLS = ["need paint", "need paint?", "need to paint?"]

In [87]:
# Acquire path for dataset (it's in a folder on an upper level)
parent_path = Path().resolve().parents[0]
json_file = parent_path / "data/final/bandai_dataset_2026-07-01.jsonl"


In [112]:
#Create df
df = pd.read_json(json_file, lines=True)

In [113]:
# Pull out image URL into separate column and drop original image column
df['image_url'] = df['image'].map(lambda x: x.get('url', x) if isinstance(x, dict) else None)
df = df.drop(columns=['image'])

In [114]:
# Pull out all infobox fields into their own columns
df = pd.concat([df.drop(['infobox'], axis=1), df['infobox'].apply(pd.Series)], axis=1)
df = df.replace(r'^\s*$', np.nan, regex=True)

In [ ]:
df.columns

In [116]:
# Check to see if rows have more than 1 paint column
conflict = df[PAINT_COLS].notna().sum(axis=1) > 1
df = df.drop(columns=['need paint']) # 'need paint' only has 1 row with this not equal to NaN, and it's not even visible on the wiki page :D
df['need_paint_clean'] = df['need paint?'].apply(normalize_paint)
df['need_to_paint_clean'] = df['need to paint?'].apply(normalize_paint)

real_conflict = (df['need_paint_clean'].notna() & df['need_to_paint_clean'].notna() & (df['need_paint_clean'] != df['need_to_paint_clean']))
print(df.loc[real_conflict, ['kit_name', 'need paint?', 'need to paint?']])

df['need_paint'] = df['need_paint_clean'].combine_first(df['need_to_paint_clean'])
df = df.drop(columns=['need paint?', 'need to paint?', 'need_paint_clean', 'need_to_paint_clean'])

                                               kit_name need paint?  \
1571  HGUC RX-0 Unicorn Gundam (Destroy Mode) (Paint...    Optional   
1965                   MG RX-78-3 G-3 Gundam (Ver. 2.0)          No   

       need to paint?  
1571              Yes  
1965  Yes (figurines)  


In [107]:
from difflib import get_close_matches
cols = list(df.columns)
for c in cols:
    matches = get_close_matches(c, cols, n=3, cutoff=0.8)
    if len(matches) > 1:
        print(c, "~", matches)

variant of ~ ['variant of', 'variant']
illustration by ~ ['illustration by', 'illustration']
variant ~ ['variant', 'variant of']
illustration ~ ['illustration', 'illustration by']


In [121]:
# Compare variant columns and combine into one
VARIANT_COL = [c for c in df.columns if 'variant' in c.lower()]
variant_conflict = df[VARIANT_COL].notna().sum(axis=1) > 1
print(f"{variant_conflict.sum()} rows with both variant columns populated")
df.loc[variant_conflict, ['kit_name','variant']]

df['variant_of'] = df.apply(merge_variants, axis=1)
df = df.drop(columns=VARIANT_COL)

1 rows with both variant columns populated


In [ ]:
# These really don't relate to the gunpla itself but rather the illustrator of the gundam or for figures + plus the random "1", "2" columns
df = df.drop(columns=['illustration by', 'image', 'sculptor','1','2','illustration','cg works by', 'finish work by','imgsize','figure sculpt','character design'])

In [ ]:
pd.set_option('display.max_rows', 200)
df['exclusive to'].unique()

In [159]:
df = df.drop(columns='availability')

In [160]:
df.columns

Index(['kit_name', 'categories', 'image_url', 'classification', 'lineup no.',
       'scale', 'franchise', 'release date', 'price', 'jan/isbn', 'need glue?',
       'materials', 'run', 'model of', 'name', 'japanese name', 'exclusive to',
       'subtitle', 'for use with', 'add-on for', 'need_paint', 'variant_of'],
      dtype='str')